Começando a ingestão de dados brutos: Camada bronze da arquitetura medalhão

In [0]:
from pyspark.sql.functions import current_timestamp

# Definição do catálogo criado e criação do banco de dados "bronze"
spark.sql("USE CATALOG medalhao")
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

# Função de ingestão
def ingerir_csv_para_bronze(nome_arquivo, nome_tabela):
    caminho_arquivo = f"/Volumes/medalhao/default/landing/{nome_arquivo}"
    
    # Lê o CSV inferindo as colunas automaticamente
    df = spark.read.csv(caminho_arquivo, header=True, inferSchema=True)
    
    # Adição da coluna timestamp_ingestion
    df_bronze = df.withColumn("timestamp_ingestion", current_timestamp())
    
    # Salva o DataFrame como uma tabela Delta no database bronze
    df_bronze.write.format("delta").mode("overwrite").saveAsTable(f"bronze.{nome_tabela}")
    
    print(f"Sucesso: {nome_arquivo} ingerido como bronze.{nome_tabela}")
    

In [0]:
# Mapeamento
ingerir_csv_para_bronze("olist_customers_dataset.csv", "tb_customers")
ingerir_csv_para_bronze("olist_geolocation_dataset.csv", "tb_geolocalizacao")
ingerir_csv_para_bronze("olist_order_items_dataset.csv", "tb_order_items")
ingerir_csv_para_bronze("olist_order_payments_dataset.csv", "tb_order_payments")
ingerir_csv_para_bronze("olist_order_reviews_dataset.csv", "tb_order_reviews")
ingerir_csv_para_bronze("olist_orders_dataset.csv", "tb_orders")
ingerir_csv_para_bronze("olist_products_dataset.csv", "tb_products")
ingerir_csv_para_bronze("olist_sellers_dataset.csv", "tb_sellers")
ingerir_csv_para_bronze("product_category_name_translation.csv", "tb_product_category_name_translation")

Sucesso: olist_customers_dataset.csv ingerido como bronze.tb_customers
Sucesso: olist_geolocation_dataset.csv ingerido como bronze.tb_geolocalizacao
Sucesso: olist_order_items_dataset.csv ingerido como bronze.tb_order_items
Sucesso: olist_order_payments_dataset.csv ingerido como bronze.tb_order_payments
Sucesso: olist_order_reviews_dataset.csv ingerido como bronze.tb_order_reviews
Sucesso: olist_orders_dataset.csv ingerido como bronze.tb_orders
Sucesso: olist_products_dataset.csv ingerido como bronze.tb_products
Sucesso: olist_sellers_dataset.csv ingerido como bronze.tb_sellers
Sucesso: product_category_name_translation.csv ingerido como bronze.tb_product_category_name_translation


In [0]:
import requests 

# Criação dos parâmetros, defini a data padrão de acordo com o os dados do dataset
dbutils.widgets.text("data_inicio", "01-01-2016")
dbutils.widgets.text("data_fim", "12-31-2018")

# Colocando os parâmetros nas variáveis
data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

# Definição do endpoint da API concatenando os parâmetros
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

# Requisição
print(f"Buscando dados de {data_inicio_formatada} até {data_fim_formatada}....")
resposta = requests.get(url)

if resposta.status_code == 200: # OK HTTP 
    dados_json = resposta.json()
    valores = dados_json.get("value") # Padrão OData como consta na URL
    
    if valores:
        df_dolar = spark.createDataFrame(valores) # Cria o df Spark 
        df_dolar_bronze = df_dolar.withColumn("timestamp_ingestion", current_timestamp()) # Adição da coluna timestamp_ingestion
        df_dolar_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze.tb_cotacao_dolar") # Salvando bronze.tb_cotacao_dolar

Buscando dados de 01-01-2016 até 12-31-2018....


In [0]:
# Listar todas as tabelas que foram criadas dentro do banco de dados bronze
print("Tabelas no schema Bronze")
display(spark.sql("SHOW TABLES IN bronze"))

# Head da tabela do dólar
print("\nAmostra de dados: tb_cotacao_dolar")
display(spark.table("bronze.tb_cotacao_dolar").limit(5))

# Head da tabela do customers
print("\nAmostra de dados: tb_customers")
display(spark.table("bronze.tb_customers").limit(5))

Tabelas no schema Bronze


database,tableName,isTemporary
bronze,tb_cotacao_dolar,false
bronze,tb_customers,false
bronze,tb_geolocalizacao,false
bronze,tb_order_items,false
bronze,tb_order_payments,false
bronze,tb_order_reviews,false
bronze,tb_orders,false
bronze,tb_product_category_name_translation,false
bronze,tb_products,false
bronze,tb_sellers,false



Amostra de dados: tb_cotacao_dolar


cotacaoCompra,dataHoraCotacao,timestamp_ingestion
4.038,2016-01-04 13:12:41.021,2026-04-07T14:23:44.198Z
4.0108,2016-01-05 13:12:41.306,2026-04-07T14:23:44.198Z
4.0297,2016-01-06 13:08:04.506,2026-04-07T14:23:44.198Z
4.0469,2016-01-07 13:07:20.817,2026-04-07T14:23:44.198Z
4.0244,2016-01-08 13:11:21.614,2026-04-07T14:23:44.198Z



Amostra de dados: tb_customers


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_name,customer_gender,customer_birth_date,customer_age,timestamp_ingestion
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Zoe Nogueira,F,1956-09-07,70,2026-04-07T14:23:00.900Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,Liam Viana,M,1974-09-23,52,2026-04-07T14:23:00.900Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,Diego Aparecida,M,1964-05-20,62,2026-04-07T14:23:00.900Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,Marcos Vinicius Azevedo,M,1967-01-14,59,2026-04-07T14:23:00.900Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,Srta. Juliana Siqueira,F,1950-08-01,76,2026-04-07T14:23:00.900Z
